# 05 · Aplicação dos cinco indicadores de qualidade

Aplicamos as mesmas fórmulas aos verbetes Antes/Depois e aos saldos das contribuições. Para os saldos dos estudantes, valores negativos são tratados como zero **antes** de calcular os indicadores. Nos grupos, primeiro somamos os saldos brutos dos integrantes e só depois aplicamos essa regra.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRUTOS = ROOT / 'dados' / 'brutos'
PROCESSADOS = ROOT / 'dados' / 'processados'
GRAFICOS = ROOT / 'resultados' / 'graficos'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)

AD = pd.read_excel(PROCESSADOS/'resultado_tcc_antes_depois_ampliado.xlsx',sheet_name='Antes_Depois_Ampliado')
PART = pd.read_excel(PROCESSADOS/'resultado_tcc_contribuicoes_ampliado.xlsx',sheet_name='Participacoes_Aluno')
GRUPOS = pd.read_excel(PROCESSADOS/'resultado_tcc_contribuicoes_ampliado.xlsx',sheet_name='Grupos_Verbetes')
BENCH = pd.read_excel(PROCESSADOS/'resultado_tcc_benchmark_artigos_top.xlsx',sheet_name='Benchmark_200')
assert (len(AD),len(PART),len(GRUPOS),len(BENCH))==(70,177,70,200)

## Parâmetros fixados pela pesquisa

As cinco dimensões são independentes; a soma de 0 a 5 é utilizada somente como síntese exploratória para ordenar os casos.

In [ ]:
PARAMETROS = {
    'bytes':25_000, 'referencias':48, 'densidade_referencias':17.4,
    'secoes':10, 'links_internos':100, 'links_externos':4,
    'categorias':4, 'imagens':3, 'densidade_imagens':1.0,
}
METRICAS=['bytes','referencias','secoes','infobox','ver_tambem','links_internos','links_externos','categorias','imagens']
DIMENSOES=['IC','IV','IEO','ICN','IRV']

def limitar(valor):
    return min(1.0,max(0.0,float(valor)))

def indicadores(m):
    b,r,s,inf,vt,li,le,c,im=[max(0.0,float(m.get(k,0) or 0)) for k in METRICAS]
    dens_ref = 10_000*r/b if b>0 else 0.0
    dens_img = 10_000*im/b if b>0 else 0.0
    escores={
        'IC':limitar(b/PARAMETROS['bytes']),
        'IV':0.60*limitar(r/PARAMETROS['referencias'])+0.40*limitar(dens_ref/PARAMETROS['densidade_referencias']),
        'IEO':0.90*limitar(s/PARAMETROS['secoes'])+0.10*limitar(inf),
        'ICN':0.50*limitar(li/PARAMETROS['links_internos'])+0.20*limitar(le/PARAMETROS['links_externos'])
              +0.20*limitar(c/PARAMETROS['categorias'])+0.10*limitar(vt),
        'IRV':0.70*limitar(im/PARAMETROS['imagens'])+0.30*limitar(dens_img/PARAMETROS['densidade_imagens']),
    }
    escores['S']=sum(escores.values())
    return escores

assert indicadores({k:0 for k in METRICAS})['S']==0
assert abs(indicadores({'bytes':25_000,'referencias':48,'secoes':10,
    'infobox':1,'ver_tambem':1,'links_internos':100,'links_externos':4,
    'categorias':4,'imagens':3})['S']-5)<1e-12

## Estados dos verbetes — Antes e Depois

In [ ]:
linhas=[]
for a in AD.itertuples(index=False):
    d=a._asdict()  # nomes preservados pela planilha
    linha={'ID unidade':d['id_unidade'],'Ano':d['ano'],'Turma':d['turma'],
           'Grupo':d['grupo'],'Tipo':d['tipo_participacao'],'Verbete':d['verbete'],
           'Criado (1=sim)':int(d['criado_pelo_grupo']),
           'N estudantes':int(d['qtd_integrantes_identificados']),
           'N edições':int(d['qtd_edicoes_grupo'])}
    for momento in ['antes','depois']:
        metr={k: d[f'{"bytes_total" if k=="bytes" else k}_{momento}'] for k in METRICAS}
        for k,v in metr.items():linha[f'{momento.title()}: {k}']=v
        for k,v in indicadores(metr).items():linha[f'{momento.title()}: {k}']=v
    for k in DIMENSOES+['S']:
        linha[f'Δ Depois−Antes: {k}']=linha[f'Depois: {k}']-linha[f'Antes: {k}']
    linhas.append(linha)
verbetes=pd.DataFrame(linhas)
assert len(verbetes)==70 and int(verbetes['Criado (1=sim)'].sum())==21

## Participações individuais — aluno + verbete

Uma pessoa pode aparecer em atividades diferentes; cada combinação aluno + verbete continua sendo um registro independente.

In [ ]:
def pontos_de_saldo(d):
    brutos={k:float(d.get(f'{k}_saldo' if k!='bytes' else 'bytes_saldo',0) or 0) for k in METRICAS}
    tratados={k:max(0.0,v) for k,v in brutos.items()}
    return brutos, tratados, indicadores(tratados)

rows=[]
for r in PART.to_dict('records'):
    bruto, tratado, scores = pontos_de_saldo(r)
    linha={'id':r['id_participacao'],'nome':r['nome_aluno'],'usuario':r['usuario_wikipedia'],
           'ano':r['ano'],'turma':r['turma'],'grupo':r['grupo'],'verbete':r['verbete'],'edicoes':r['edicoes']}
    linha.update({f'Saldo bruto: {k}':v for k,v in bruto.items()})
    linha.update({f'Saldo tratado: {k}':v for k,v in tratado.items()})
    linha.update(scores)
    rows.append(linha)
participacoes=pd.DataFrame(rows)
assert len(participacoes)==177 and participacoes['id'].is_unique

## Grupos/verbetes — soma dos integrantes

Não somamos escores individuais. Somamos os **saldos brutos** dos integrantes de cada grupo/verbete e aplicamos novamente as cinco fórmulas ao saldo conjunto.

In [ ]:
rows=[]
for r in GRUPOS.to_dict('records'):
    bruto, tratado, scores=pontos_de_saldo(r)
    grupo=r['grupo']
    id_unidade=''  # o ID oficial é recuperado logo abaixo pelo cruzamento da chave
    linha={'id':id_unidade,'ano':r['ano'],'turma':r['turma'],'grupo':grupo,
           'tipo':'grupo','verbete':r['verbete'], 'n':r['participacoes_identificadas']}
    linha.update({f'Saldo bruto: {k}':v for k,v in bruto.items()})
    linha.update({f'Saldo tratado: {k}':v for k,v in tratado.items()})
    linha.update(scores)
    rows.append(linha)
grupos=pd.DataFrame(rows)
assert len(grupos)==70
# Alguns casos têm chave de grupo específica; recuperamos o ID histórico pela chave de análise.
chave=['ano','turma','grupo','verbete']
# Em duas unidades individuais o campo 'grupo' é vazio na base Antes/Depois.
# Por isso o vínculo usa ano, turma e verbete (combinação única no recorte).
chave_vinculo=['ano','turma','verbete']
assert not AD.duplicated(chave_vinculo).any()
ids=AD[chave_vinculo+['id_unidade','tipo_participacao']].rename(columns={'id_unidade':'id_correto'})
grupos=grupos.merge(ids,on=chave_vinculo,how='left',validate='one_to_one')
assert grupos['id_correto'].notna().all()
grupos['id']=grupos.pop('id_correto')
grupos['tipo']=grupos.pop('tipo_participacao')

## Conferência e rankings

Todos os escores individuais ficam de 0 a 1; a soma exploratória fica entre 0 e 5. Não existe ranking de pessoas acumulado entre verbetes.

In [ ]:
for dataframe in [verbetes,participacoes,grupos]:
    colunas=[c for c in dataframe if c in DIMENSOES or c in ['Antes: S','Depois: S','S']]
    # Deltas Antes/Depois podem ser negativos e não fazem parte dessa validação.
    assert dataframe[colunas].apply(pd.to_numeric).ge(-1e-10).all().all()
    assert dataframe[colunas].apply(pd.to_numeric).le(5+1e-10).all().all()

final=verbetes.sort_values('Depois: S',ascending=False).head(20)
evolucao=verbetes.sort_values('Δ Depois−Antes: S',ascending=False).head(20)
individuais=participacoes.sort_values('S',ascending=False).head(20)
coletivos=grupos.sort_values('S',ascending=False).head(20)
print('Média Antes/Depois:',verbetes['Antes: S'].mean().round(3),verbetes['Depois: S'].mean().round(3))
print('Top verbete final:',final.iloc[0]['Verbete'])

## Exportação dos resultados

In [ ]:
parametros=pd.DataFrame([{'Parâmetro':k,'Valor':v} for k,v in PARAMETROS.items()])
saida=PROCESSADOS/'Resultados_Indicadores_TCC_revisado.xlsx'
with pd.ExcelWriter(saida,engine='openpyxl') as writer:
    parametros.to_excel(writer,sheet_name='Parametros',index=False)
    verbetes.to_excel(writer,sheet_name='Verbetes_70',index=False)
    participacoes.to_excel(writer,sheet_name='Participacoes_177',index=False)
    grupos.to_excel(writer,sheet_name='Grupos_70',index=False)
    final.to_excel(writer,sheet_name='Top20_Final',index=False)
    evolucao.to_excel(writer,sheet_name='Top20_Evolucao',index=False)
    individuais.to_excel(writer,sheet_name='Top20_Participacoes',index=False)
    coletivos.to_excel(writer,sheet_name='Top20_Grupos',index=False)
print('Arquivo gerado:',saida.name)